In [5]:
import pandas as pd
import numpy as np

# Load Dataset
df = pd.read_excel(r"C:\Users\User\Downloads\archive\Telco_customer_churn-Copy1.xlsx")

# Create Cleaning Log
cleaning_log = []

# --------------------------------------------------
# 1. Handle Missing Values
# --------------------------------------------------

missing_before = df.isnull().sum().sum()

# Fill categorical missing values with mode
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Fill numerical missing values with median
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

missing_after = df.isnull().sum().sum()

cleaning_log.append(
    f"Missing values handled: {missing_before} -> {missing_after}"
)

# --------------------------------------------------
# 2. Remove Duplicates
# --------------------------------------------------

duplicates_before = df.duplicated().sum()

df.drop_duplicates(inplace=True)

duplicates_after = df.duplicated().sum()

cleaning_log.append(
    f"Duplicates removed: {duplicates_before - duplicates_after}"
)

# --------------------------------------------------
# 3. Fix Data Types
# --------------------------------------------------

# Convert object columns with few unique values to category
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() < 20:
        df[col] = df[col].astype('category')

cleaning_log.append(
    "Converted low-cardinality object columns to category datatype"
)

# --------------------------------------------------
# 4. Handle Outliers using IQR Method
# --------------------------------------------------

numeric_cols = df.select_dtypes(include=['int64','float64']).columns

rows_before = len(df)

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

rows_after = len(df)

cleaning_log.append(
    f"Outliers removed using IQR method: {rows_before - rows_after} rows removed"
)

# --------------------------------------------------
# 5. Standardize Column Names
# --------------------------------------------------

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(' ', '_')
)

cleaning_log.append(
    "Column names standardized to lowercase with underscores"
)

# --------------------------------------------------
# 6. Save Cleaned Dataset
# --------------------------------------------------

df.to_excel("cleaned_telco_customer_churn.xlsx", index=False)

cleaning_log.append(
    "Cleaned dataset saved as cleaned_telco_customer_churn.xlsx"
)

# --------------------------------------------------
# 7. Display Cleaning Log
# --------------------------------------------------

print("\nDATA CLEANING LOG")
print("-" * 50)

for step, log in enumerate(cleaning_log, start=1):
    print(f"{step}. {log}")

print("\nFinal Dataset Shape:", df.shape)

C:\Users\User\AppData\Local\Temp\ipykernel_30112\264842097.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)



DATA CLEANING LOG
--------------------------------------------------
1. Missing values handled: 5174 -> 0
2. Duplicates removed: 0
3. Converted low-cardinality object columns to category datatype
4. Outliers removed using IQR method: 0 rows removed
5. Column names standardized to lowercase with underscores
6. Cleaned dataset saved as cleaned_telco_customer_churn.xlsx

Final Dataset Shape: (7043, 33)
